# Tweet Sentiment Extraction on Google Colab

Before running:
1. Upload the whole `tse` folder to **My Drive** in Google Drive. It already has the data in `tse/data`.
2. In Colab: **Runtime > Change runtime type > T4 GPU**.
3. Run the cells in order. If Colab disconnects, run them again: finished folds are skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Settings

Each level-one model is about 500 MB and the full setup has 25 (5 seeds x 5 folds).
- **Option A (default):** keep the models on Drive in half size, about 250 MB each (about 6 GB for all 25).
- **Option B:** set `MODELS = '/content/models'` to keep them off Drive. Then run `train1` and `test1` for a seed
  in the same session, because Colab's disk is wiped when the session ends.

In [ ]:
import os
PROJECT = '/content/drive/MyDrive/tse'          # where you uploaded the folder
DATA = f'{PROJECT}/data'
WORK = f'{PROJECT}/work'                         # results stay on Drive
PRETRAINED = f'{PROJECT}/roberta-base-squad2'    # downloaded once, then reused
MODELS = f'{WORK}/models'                        # Option B: '/content/models'
HALF = '--save-fp16-weights'                     # '' to save full-size models

SEEDS = '0'   # one seed (5 folds) per session is safe; next time '1', then '2', ...
ARGS = (f'--data-dir "{DATA}" --work-dir "{WORK}" --models-dir "{MODELS}" '
        f'--pretrained "{PRETRAINED}" --num-workers 2')

assert os.path.exists(f'{PROJECT}/main.py'), 'tse folder not found on Drive'
assert os.path.exists(f'{DATA}/train.csv'), 'data/train.csv not found'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# RoBERTa checkpoint used in 2021 (about 500 MB, downloaded only the first time)
if not os.path.exists(f'{PRETRAINED}/config.json'):
    from huggingface_hub import snapshot_download
    snapshot_download('deepset/roberta-base-squad2', local_dir=PRETRAINED,
                      allow_patterns=['config.json', 'vocab.json', 'merges.txt',
                                      'model.safetensors'])
!ls -la "{PRETRAINED}"

## Level one for the chosen seed(s): training, then predictions on test.csv

In [ ]:
!cd "{PROJECT}" && python main.py train1 {ARGS} --seeds {SEEDS} --fp16 {HALF}

In [ ]:
!cd "{PROJECT}" && python main.py test1 {ARGS} --seeds {SEEDS} --fp16

## Level two and the submission

This uses every seed that has finished `train1` and `test1`, so you can run it after seed 0
and again after more seeds are done. Level two is small; it does not need a long GPU session.

In [ ]:
# uses the seeds that are ready; add --seeds 0 1 to choose them yourself
!cd "{PROJECT}" && python main.py train2 {ARGS}
!cd "{PROJECT}" && python main.py predict {ARGS}

In [ ]:
import json, pandas as pd
print(json.dumps(json.load(open(f'{WORK}/scores.json')), indent=2))
pd.read_csv(f'{WORK}/submission.csv').head(10)

## Try the demo

In [ ]:
!cd "{PROJECT}" && python demo.py --pretrained "{PRETRAINED}" --work-dir "{WORK}" --models-dir "{MODELS}" \
    --device cuda --text "Sooo SAD I will miss you here in San Diego!!!" --sentiment negative